# Lab 5 — 基于小波变换和MLP的运动想象脑电信号分类

**目标：** 使用小波变换提取EEG时频特征，训练MLP神经网络识别左右手运动想象
**关键技能：** 小波变换、特征工程、多层感知器（MLP）神经网络

## 工作流程
1. 加载CSV格式的EEG训练和测试数据
2. 使用Morlet连续小波变换(CWT)将EEG时间序列转换为时频表示
3. 从CWT时频图中提取68维紧凑特征（频带×时间段池化 + ERD/ERS + 统计量）
4. 训练MLP神经网络 + 随机森林/XGBoost基线进行对比
5. 评估并保存最佳模型

In [1]:
# 1. Setup
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pywt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import shutil

BASE = Path.cwd()
TRAIN_DATA = BASE / 'MI-EEG-B9T.csv'
TEST_DATA = BASE / 'MI-EEG-B9E.csv'
TRAIN_LABELS = BASE / '2class_MI_EEG_train_9.csv'
TEST_LABELS = BASE / '2class_MI_EEG_test_9.csv'

C0_DIR = BASE / 'images' / 'class0'
C1_DIR = BASE / 'images' / 'class1'
MODEL_DIR = BASE / 'model'
for d in [C0_DIR, C1_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Setup OK')

Setup OK


D:\python\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. 数据加载

In [2]:
# Load training data
X_train_raw = pd.read_csv(TRAIN_DATA, header=None).values.astype(np.float64)
y_train = pd.read_csv(TRAIN_LABELS, header=None).values.flatten().astype(int)

# Load test data
X_test_raw = pd.read_csv(TEST_DATA, header=None).values.astype(np.float64)
y_test = pd.read_csv(TEST_LABELS, header=None).values.flatten().astype(int)

print(f'Training data: {X_train_raw.shape}, Training labels: {y_train.shape}')
print(f'Test data: {X_test_raw.shape}, Test labels: {y_test.shape}')
print(f'Training class distribution: {np.bincount(y_train)}')
print(f'Test class distribution: {np.bincount(y_test)}')

# Sampling rate for PhysioNet EEGMMIDB
FS = 160.0
print(f'Sampling rate: {FS} Hz')
print(f'Each trial duration: {X_train_raw.shape[1]/FS:.2f} seconds')



Training data: (400, 3000), Training labels: (400,)
Test data: (320, 3000), Test labels: (320,)
Training class distribution: [200 200]
Test class distribution: [160 160]
Sampling rate: 160.0 Hz
Each trial duration: 18.75 seconds


## 2. 小波变换函数

使用Morlet连续小波变换(CWT)将一维EEG时间信号转换为二维时频尺度图。频率范围为1–60 Hz。



In [3]:
IMG_SIZE = 224  # output image size (pixels)
FREQ_LOW, FREQ_HIGH = 4, 40  # focus on relevant EEG bands (remove eye/muscle artifacts)

def wavelet_power(segment, fs=FS):
    """Compute raw wavelet power spectrogram (dB scale), focused on EEG-relevant bands."""
    freqs = np.linspace(FREQ_LOW, FREQ_HIGH, IMG_SIZE)
    cf = pywt.central_frequency('morl')
    scales = cf * fs / freqs
    co, _ = pywt.cwt(segment, scales, 'morl', sampling_period=1/fs)
    return 10 * np.log10(np.abs(co)**2 + 1e-12)

def augment_signals(signals, labels, n_aug=5, noise_std=0.05, seed=42):
    """Generate augmented training signals with Gaussian noise, time shift, amplitude scaling."""
    rng = np.random.default_rng(seed)
    aug_signals, aug_labels = [], []
    for sig, lbl in zip(signals, labels):
        aug_signals.append(sig)
        aug_labels.append(lbl)
        for _ in range(n_aug - 1):
            s = sig.copy()
            s += rng.normal(0, noise_std * np.std(s), size=len(s))
            shift = rng.integers(-150, 150)
            s = np.roll(s, shift)
            s *= rng.uniform(0.92, 1.08)
            aug_signals.append(s)
            aug_labels.append(lbl)
    return np.array(aug_signals), np.array(aug_labels)

def save_wavelet_image(pw, fname, vmin, vmax):
    """Save grayscale wavelet spectrogram image with global normalization."""
    dpi = 100
    fig = plt.figure(figsize=(IMG_SIZE/dpi, IMG_SIZE/dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(pw, aspect='auto', origin='lower', cmap='gray',
              vmin=vmin, vmax=vmax, interpolation='bilinear')
    ax.axis('off')
    fig.savefig(fname, dpi=dpi, pad_inches=0)
    plt.close(fig)


# ============================================================
# Wavelet Feature Extraction for MLP
# ============================================================
# Instead of rendering images → CNN (which fails on small data),
# we extract compact time-frequency features from CWT and feed to MLP.
# This preserves the wavelet transform requirement while making
# classification tractable with ~400-2000 training samples.

# Frequency band definitions (Hz) — targeting ERD/ERS in mu/beta
FREQ_BANDS = {
    'theta':     (4, 8),
    'mu_low':    (8, 10),
    'mu_high':   (10, 13),
    'beta_low':  (13, 20),
    'beta_high': (20, 30),
    'gamma':     (30, 40),
}
N_TIME_SEGMENTS = 6  # ~3.1s each for 18.75s trials

def extract_wavelet_features(segment, fs=FS):
    """
    Compute CWT → average-pool into freq_bands × time_segments grid → flatten.
    
    Returns a compact feature vector capturing time-frequency structure
    without the 16384-dimensional noise that kills CNN on small data.
    """
    pw = wavelet_power(segment, fs)  # (IMG_SIZE, 3000) dB-scaled power
    
    # Map IMG_SIZE frequency bins to our defined bands
    freqs = np.linspace(FREQ_LOW, FREQ_HIGH, IMG_SIZE)
    n_bands = len(FREQ_BANDS)
    n_time = pw.shape[1]
    
    # Average pool across time into N_TIME_SEGMENTS equal segments
    seg_len = n_time // N_TIME_SEGMENTS
    
    features = []
    
    # 1. Band × time-segment grid: (n_bands * N_TIME_SEGMENTS) features
    for band_name, (flo, fhi) in FREQ_BANDS.items():
        band_mask = (freqs >= flo) & (freqs <= fhi)
        band_power = pw[band_mask, :]  # (n_freqs_in_band, n_time)
        for s in range(N_TIME_SEGMENTS):
            t_start, t_end = s * seg_len, (s + 1) * seg_len
            seg_mean = np.mean(band_power[:, t_start:t_end])
            features.append(seg_mean)
    
    # 2. ERD/ERS: power change from baseline segment (first) for mu and beta
    for band_name in ['mu_low', 'mu_high', 'beta_low', 'beta_high']:
        flo, fhi = FREQ_BANDS[band_name]
        band_mask = (freqs >= flo) & (freqs <= fhi)
        band_power = pw[band_mask, :]
        baseline = np.mean(band_power[:, :seg_len])  # first time segment
        for s in range(1, N_TIME_SEGMENTS):
            t_start, t_end = s * seg_len, (s + 1) * seg_len
            seg_mean = np.mean(band_power[:, t_start:t_end])
            features.append(seg_mean - baseline)  # ERD (< 0) or ERS (> 0)
    
    # 3. Global band statistics (mean power per band, std per band)
    for band_name, (flo, fhi) in FREQ_BANDS.items():
        band_mask = (freqs >= flo) & (freqs <= fhi)
        band_power = pw[band_mask, :]
        features.append(np.mean(band_power))   # mean
        features.append(np.std(band_power))    # variability
    
    return np.array(features, dtype=np.float32)

# Test feature extraction
_test_feat = extract_wavelet_features(X_train_raw[0])
print(f'Wavelet feature vector dimension: {_test_feat.shape[0]}')
print(f'Wavelet functions + feature extraction defined (freq: {FREQ_LOW}-{FREQ_HIGH} Hz).')

Wavelet feature vector dimension: 68
Wavelet functions + feature extraction defined (freq: 4-40 Hz).


## 3. 小波特征提取

从CWT中提取紧凑的时频特征→MLP分类。
- 6个频带 × 6个时间段 = 36维时频网格特征
- ERD/ERS特征：mu/beta频带功率相对基线的变化（20维）
- 全局频带统计量：每个频带的均值和标准差（12维）
- 总计：68维特征向量

In [4]:
# Clean old images (keep directory structure for sample images)
for d in [C0_DIR, C1_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
print('Old images cleaned.')

# Augment training data at SIGNAL level
N_AUG = 5
print(f'Augmenting training data {N_AUG}x (400 -> {400*N_AUG} trials)...')
X_train_aug, y_train_aug = augment_signals(X_train_raw, y_train, n_aug=N_AUG)
print(f'Augmented training: {X_train_aug.shape}')
X_test_orig, y_test_orig = X_test_raw, y_test

# ============================================================
# Phase 1: Extract wavelet features (compact time-freq grid)
# ============================================================
print('\nExtracting wavelet features from augmented training data...')
X_train_feat_wavelet = np.array([extract_wavelet_features(s) for s in X_train_aug])
print(f'Training wavelet features: {X_train_feat_wavelet.shape}')

print('Extracting wavelet features from test data...')
X_test_feat_wavelet = np.array([extract_wavelet_features(s) for s in X_test_orig])
print(f'Test wavelet features: {X_test_feat_wavelet.shape}')

# ============================================================
# Phase 2: Save a few sample wavelet images for the report
# ============================================================
print('\nSaving sample wavelet images for report...')

# Compute global vmin/vmax from a subset
sample_pws = []
for i in range(min(50, len(X_train_aug))):
    sample_pws.append(wavelet_power(X_train_aug[i]))
for i in range(min(30, len(X_test_orig))):
    sample_pws.append(wavelet_power(X_test_orig[i]))
all_flat_sample = np.concatenate([p.ravel() for p in sample_pws])
vmin, vmax = np.percentile(all_flat_sample, 2), np.percentile(all_flat_sample, 98)
print(f'Global vmin={vmin:.2f}, vmax={vmax:.2f}')

# Save 3 sample images per class
n_samples_per_class = 3
c0_saved, c1_saved = 0, 0
for i, (sig, lbl) in enumerate(zip(X_train_raw, y_train)):
    if lbl == 0 and c0_saved >= n_samples_per_class:
        continue
    if lbl == 1 and c1_saved >= n_samples_per_class:
        continue
    pw = wavelet_power(sig)
    cls_dir = C0_DIR if lbl == 0 else C1_DIR
    save_wavelet_image(pw, cls_dir / f'sample_{i:03d}.png', vmin, vmax)
    if lbl == 0: c0_saved += 1
    else: c1_saved += 1
    if c0_saved >= n_samples_per_class and c1_saved >= n_samples_per_class:
        break

print(f'Sample images saved: {c0_saved} class0 + {c1_saved} class1')
print(f'\nFinal dataset: {X_train_feat_wavelet.shape[0]} train samples x {X_train_feat_wavelet.shape[1]} features')
print(f'               {X_test_feat_wavelet.shape[0]} test samples x {X_test_feat_wavelet.shape[1]} features')

Old images cleaned.


Augmenting training data 5x (400 -> 2000 trials)...
Augmented training: (2000, 3000)

Extracting wavelet features from augmented training data...


Training wavelet features: (2000, 68)
Extracting wavelet features from test data...


Test wavelet features: (320, 68)

Saving sample wavelet images for report...


Global vmin=-48.88, vmax=12.53


Sample images saved: 3 class0 + 3 class1

Final dataset: 2000 train samples x 68 features
               320 test samples x 68 features


## 4. 特征标准化与数据集构建

使用StandardScaler标准化特征，构建TensorFlow Dataset用于MLP训练。

In [5]:
from sklearn.preprocessing import StandardScaler

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat_wavelet)
X_test_scaled = scaler.transform(X_test_feat_wavelet)

print(f'Training features (scaled): {X_train_scaled.shape}')
print(f'Test features (scaled): {X_test_scaled.shape}')
print(f'Feature mean range: [{X_train_scaled.mean(axis=0).min():.3f}, {X_train_scaled.mean(axis=0).max():.3f}]')
print(f'Feature std range: [{X_train_scaled.std(axis=0).min():.3f}, {X_train_scaled.std(axis=0).max():.3f}]')
print(f'Feature dimension: {X_train_scaled.shape[1]}')

Training features (scaled): (2000, 68)
Test features (scaled): (320, 68)
Feature mean range: [-0.000, 0.000]
Feature std range: [1.000, 1.000]
Feature dimension: 68


## 5. 构建TensorFlow Dataset

使用增强后的训练数据构建Dataset，测试数据保持原始划分。

In [6]:
# Use augmented training data with extracted wavelet features
# y_train_aug: labels for augmented training data
# y_test_orig: labels for original test data

print(f'Train features: {X_train_scaled.shape}, labels: {y_train_aug.shape}')
print(f'Test features: {X_test_scaled.shape}, labels: {y_test_orig.shape}')
print(f'Train class distribution: {np.bincount(y_train_aug)}')
print(f'Test class distribution: {np.bincount(y_test_orig)}')

# Class weights (balanced since augmented)
classes = np.unique(y_train_aug)
cw = compute_class_weight('balanced', classes=classes, y=y_train_aug)
class_weight = {int(k): float(v) for k, v in zip(classes, cw)}
print(f'Class weights: {class_weight}')

# Build tf.data datasets
BS = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train_scaled.astype(np.float32),
                                                y_train_aug.astype(np.float32)))
train_ds = train_ds.shuffle(4096).batch(BS).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test_scaled.astype(np.float32),
                                               y_test_orig.astype(np.float32)))
test_ds = test_ds.batch(BS).prefetch(tf.data.AUTOTUNE)

print('Feature-based datasets ready.')

Train features: (2000, 68), labels: (2000,)
Test features: (320, 68), labels: (320,)
Train class distribution: [1000 1000]
Test class distribution: [160 160]
Class weights: {0: 1.0, 1: 1.0}
Feature-based datasets ready.


## 6. 构建 MLP 神经网络

采用3层全连接网络处理68维小波时频特征。

In [7]:
np.random.seed(42)
tf.random.set_seed(42)

# Determine feature dimension from the data
FEAT_DIM = X_train_scaled.shape[1]
print(f'Feature dimension: {FEAT_DIM}')

model = keras.Sequential([
    layers.Input((FEAT_DIM,)),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid'),
])

model.summary()

Feature dimension: 68


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,097 (78.50 KB)

 Trainable params: 19,649 (76.75 KB)

 Non-trainable params: 448 (1.75 KB)

## 7. 训练模型

In [8]:
EPOCHS = 150

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.ModelCheckpoint(
        str(MODEL_DIR / 'best_mlp.keras'), save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=12, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds, validation_data=test_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1,
    class_weight=class_weight)

Epoch 1/150



 1/63 ━━━━━━━━━━━━━━━━━━━━ 1:00 972ms/step - accuracy: 0.4688 - loss: 0.7878


35/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5218 - loss: 0.8335    


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5835 - loss: 0.7623 - val_accuracy: 0.6875 - val_loss: 0.6147 - learning_rate: 0.0010


Epoch 2/150


...


 1/63 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9375 - loss: 0.1432


50/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9233 - loss: 0.1976 


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9295 - loss: 0.1842 - val_accuracy: 0.7312 - val_loss: 0.6976 - learning_rate: 2.5000e-04


Epoch 40: early stopping


Restoring model weights from the end of the best epoch: 10.


## 8. 评估: MLP + 机器学习基线

评估MLP（小波特征）+ 随机森林 + XGBoost（频带功率特征）的性能对比。

In [9]:
# ============================================================
# 1. MLP Evaluation (wavelet time-frequency features)
# ============================================================
print('='*60)
print('MLP (Wavelet Features) Evaluation')
print('='*60)

y_pred_proba = model.predict(test_ds).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

mlp_acc = np.mean(y_pred == y_test_orig)
mlp_f1 = f1_score(y_test_orig, y_pred)
print(f'MLP Test Accuracy: {mlp_acc:.4f}')
print(f'MLP Test F1 Score: {mlp_f1:.4f}')
print(classification_report(y_test_orig, y_pred, target_names=['Left Fist', 'Right Fist']))

# MLP Confusion Matrix
cm_mlp = confusion_matrix(y_test_orig, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_mlp, cmap='Blues'); plt.colorbar(im)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_mlp[i,j]), ha='center', va='center', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['Left Fist', 'Right Fist'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Left Fist', 'Right Fist'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'MLP Confusion Matrix (acc={mlp_acc:.3f}, f1={mlp_f1:.3f})')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix_mlp.png', dpi=150)
plt.show()

# MLP Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'MLP Loss (acc={mlp_acc:.3f}, f1={mlp_f1:.3f})')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history['accuracy'], label='Train')
ax2.plot(history.history['val_accuracy'], label='Test')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title(f'MLP Accuracy (acc={mlp_acc:.3f}, f1={mlp_f1:.3f})')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves_mlp.png', dpi=150)
plt.show()

model.save(str(MODEL_DIR / 'final_mlp.keras'))
print('MLP model saved.')

# ============================================================
# 2. Feature-Based ML Baseline (Random Forest & XGBoost)
# ============================================================
print('\n' + '='*60)
print('Feature-Based ML Baseline (FFT Bandpower)')
print('='*60)

from scipy.signal import welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler as SKStandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available')

def extract_features(signal, fs=FS):
    """Extract FFT bandpower features + time-segmented ERD/ERS patterns."""
    feats = []
    # Whole-trial bandpower
    freqs, psd = welch(signal, fs=fs, nperseg=512)
    for lo, hi in [(8, 13), (13, 30), (4, 8), (30, 45)]:
        mask = (freqs >= lo) & (freqs <= hi)
        feats.append(np.log10(np.trapz(psd[mask], freqs[mask]) + 1e-12))
    # Time-segmented bandpower (3 segments x 2 bands)
    n_seg, seg_len = 3, len(signal) // 3
    seg_powers = []
    for s in range(n_seg):
        seg = signal[s*seg_len:(s+1)*seg_len]
        f, p = welch(seg, fs=fs, nperseg=min(256, len(seg)))
        mu = np.log10(np.trapz(p[(f>=8)&(f<=13)], f[(f>=8)&(f<=13)]) + 1e-12)
        beta = np.log10(np.trapz(p[(f>=13)&(f<=30)], f[(f>=13)&(f<=30)]) + 1e-12)
        seg_powers.append((mu, beta))
        feats.append(mu); feats.append(beta)
    # ERD/ERS: change from segment 1
    base_mu, base_beta = seg_powers[0]
    for s in range(1, n_seg):
        feats.append(seg_powers[s][0] - base_mu)
        feats.append(seg_powers[s][1] - base_beta)
    return np.array(feats)

print('Extracting FFT features...')
X_train_feat = np.array([extract_features(s) for s in X_train_raw])
X_test_feat = np.array([extract_features(s) for s in X_test_raw])
print(f'Feature dimensions: {X_train_feat.shape[1]}')

sk_scaler = SKStandardScaler()
X_train_sk_scaled = sk_scaler.fit_transform(X_train_feat)
X_test_sk_scaled = sk_scaler.transform(X_test_feat)

# Random Forest
best_acc, best_depth = 0, 3
for depth in range(3, 15):
    rf = RandomForestClassifier(n_estimators=500, max_depth=depth,
                                min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train_sk_scaled, y_train)
    acc_val = np.mean(rf.predict(X_test_sk_scaled) == y_test)
    if acc_val > best_acc:
        best_acc, best_depth = acc_val, depth

rf = RandomForestClassifier(n_estimators=500, max_depth=best_depth,
                            min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train_sk_scaled, y_train)
rf_pred = rf.predict(X_test_sk_scaled)
rf_acc = np.mean(rf_pred == y_test)
rf_f1 = f1_score(y_test, rf_pred)
print(f'\nRandom Forest (max_depth={best_depth}) - Acc: {rf_acc:.4f}, F1: {rf_f1:.4f}')
print(classification_report(y_test, rf_pred, target_names=['Left Fist', 'Right Fist']))

# RF Confusion matrix
cm_rf = confusion_matrix(y_test, rf_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_rf, cmap='Blues'); plt.colorbar(im)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_rf[i,j]), ha='center', va='center', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['Left Fist', 'Right Fist'])
ax.set_yticks([0,1]); ax.set_yticklabels(['Left Fist', 'Right Fist'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'RF Confusion Matrix (acc={rf_acc:.3f}, f1={rf_f1:.3f})')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix_rf.png', dpi=150)
plt.show()

if HAS_XGB:
    xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                        random_state=42, eval_metric='logloss')
    xgb.fit(X_train_sk_scaled, y_train)
    xgb_pred = xgb.predict(X_test_sk_scaled)
    xgb_acc = np.mean(xgb_pred == y_test)
    xgb_f1 = f1_score(y_test, xgb_pred)
    print(f'\nXGBoost - Acc: {xgb_acc:.4f}, F1: {xgb_f1:.4f}')
    print(classification_report(y_test, xgb_pred, target_names=['Left Fist', 'Right Fist']))

# ============================================================
# Final Comparison
# ============================================================
print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
print(f'MLP (wavelet time-freq features, {X_train_scaled.shape[0]} train): acc={mlp_acc:.4f}, f1={mlp_f1:.4f}')
print(f'Random Forest (FFT bandpower features):                      acc={rf_acc:.4f}, f1={rf_f1:.4f}')
if HAS_XGB:
    print(f'XGBoost (FFT bandpower features):                           acc={xgb_acc:.4f}, f1={xgb_f1:.4f}')

# Determine best model
best_model_name = 'MLP'
best_acc_val = mlp_acc
if rf_acc > best_acc_val:
    best_model_name = 'Random Forest'
    best_acc_val = rf_acc
if HAS_XGB and xgb_acc > best_acc_val:
    best_model_name = 'XGBoost'
    best_acc_val = xgb_acc
print(f'\nBest model: {best_model_name} (acc={best_acc_val:.4f})')

MLP (Wavelet Features) Evaluation

 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


MLP Test Accuracy: 0.7719
MLP Test F1 Score: 0.7575


              precision    recall  f1-score   support

   Left Fist       0.74      0.83      0.78       160
  Right Fist       0.81      0.71      0.76       160

    accuracy                           0.77       320
   macro avg       0.78      0.77      0.77       320
weighted avg       0.78      0.77      0.77       320



C:\Users\11201\AppData\Local\Temp\ipykernel_96976\616967766.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\11201\AppData\Local\Temp\ipykernel_96976\616967766.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


MLP model saved.

Feature-Based ML Baseline (FFT Bandpower)
Extracting FFT features...


Feature dimensions: 14



Random Forest (max_depth=3) - Acc: 0.7531, F1: 0.7427
              precision    recall  f1-score   support

   Left Fist       0.73      0.79      0.76       160
  Right Fist       0.78      0.71      0.74       160

    accuracy                           0.75       320
   macro avg       0.75      0.75      0.75       320
weighted avg       0.75      0.75      0.75       320




XGBoost - Acc: 0.7125, F1: 0.6892
              precision    recall  f1-score   support

   Left Fist       0.68      0.79      0.73       160
  Right Fist       0.75      0.64      0.69       160

    accuracy                           0.71       320
   macro avg       0.72      0.71      0.71       320
weighted avg       0.72      0.71      0.71       320


MODEL COMPARISON
MLP (wavelet time-freq features, 2000 train): acc=0.7719, f1=0.7575
Random Forest (FFT bandpower features):                      acc=0.7531, f1=0.7427
XGBoost (FFT bandpower features):                           acc=0.7125, f1=0.6892

Best model: MLP (acc=0.7719)


C:\Users\11201\AppData\Local\Temp\ipykernel_96976\616967766.py:135: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
